In [ ]:
import os, re, glob, json, time, textwrap, ast
import pandas as pd
import pdfplumber
import requests

# ======================= CONFIG (EDIT FOR EACH BRAND) ========================

BRAND          = "Hanes"                        
BRAND_ALIASES  = ["Hanes", "Hanesbrands"]       

BASE_DIR       = r"./data/Brands_Articles"                    
BRANDS_ROOT    = os.path.join(BASE_DIR, "Brands_Articles")
BRAND_PDF_DIR  = os.path.join(BRANDS_ROOT, BRAND)   

# Global media->country file
MEDIA_COUNTRY_GLOBAL = os.path.join(BASE_DIR, "Rana_Plaza", "factiva_titlelist_with_countries.xlsx")

# Year filter
YEAR_FILTER = 2013

# Brand-specific outputs
BRAND_LONG_CSV     = os.path.join(BRAND_PDF_DIR, f"{BRAND}_factiva_parsed_tags_long.csv")
BRAND_WIDE_CSV     = os.path.join(BRAND_PDF_DIR, f"{BRAND}_factiva_parsed_tags_wide.csv")
BRAND_RELEV_CSV    = os.path.join(BRAND_PDF_DIR, f"{BRAND}_relevance_labels.csv")
BRAND_CACHE_JSON   = os.path.join(BRAND_PDF_DIR, f"{BRAND}_relevance_cache.json")
BRAND_COUNTRY_CSV  = os.path.join(BRAND_PDF_DIR, f"{BRAND}_country_month_counts.csv")
BRAND_FINAL_XLSX   = os.path.join(BRAND_PDF_DIR, f"{BRAND}_final_database.xlsx")

# Factiva Tag Regex (Start of line, 2-4 uppercase letters, followed by space or end of line)
TAG_LINE_RE = re.compile(r"^([A-Z]{2,4})(\s+.*)?$")
# Skip metadata lines like "Page 1 of 100"
PAGE_FOOTER_RE = re.compile(r"Page \d+ of \d+.*Factiva", re.IGNORECASE)

# Language -> "obvious" audience country mapping
LANG_TO_COUNTRY = {
    "danish": "Denmark",
    "finnish": "Finland",
    "greek": "Greece",
    "hungarian": "Hungary",
    "polish": "Poland",
    "czech": "Czech Republic",
    "romanian": "Romania",
    "swedish": "Sweden",
    "norwegian": "Norway",
    "icelandic": "Iceland",
}

# Ollama config (local LLM)
OLLAMA_URL    = "http://localhost:11434/api/generate"
OLLAMA_MODEL  = "mistral"
TIMEOUT_SEC   = 120
SLEEP_BETWEEN = 0.05 


# ====================== PARSING LOGIC (NEW & FIXED) ==========================

def parse_pdf_linear(path):
    """
    Robust linear parser for Factiva PDFs.
    Reads text line-by-line. If a line starts with a TAG (e.g., HD, TD), 
    it starts a new field. Otherwise, it appends to the current field.
    """
    rows = []
    
    with pdfplumber.open(path) as pdf:
        article_idx = -1
        # "cur_tag" is the tag we are currently reading (e.g., 'TD')
        # "cur_val" is a list of strings belonging to that tag
        cur_tag, cur_val = None, []
        
        # We need to detect the start of a new article. 
        # Usually, 'HD' (Headline) or 'SE' (Section) resets the context if we were reading other tags.
        # A simple heuristic: if we hit HD, we increment article_idx.
        
        for page in pdf.pages:
            # extract_text(x_tolerance=...) helps merge letters into words better
            text = page.extract_text(x_tolerance=2, y_tolerance=3)
            if not text:
                continue
                
            lines = text.split('\n')
            
            for line in lines:
                line = line.strip()
                if not line: 
                    continue
                if PAGE_FOOTER_RE.search(line):
                    continue

                # Check if this line looks like a TAG start
                # e.g. "HD     Headline Text" or just "HD"
                match = TAG_LINE_RE.match(line)
                
                # We treat it as a tag ONLY if the tag part is a known Factiva code
                # Common codes: HD, TD, SN, PD, WC, SC, CO, IN, NS, RE, IPD, PUB, AN
                is_tag = False
                if match:
                    code_cand = match.group(1)
                    # Filter out false positives (like "THE" or "AND" at start of line)
                    if code_cand in {"HD","TD","SN","PD","WC","SC","CO","IN","NS","RE","IPD","PUB","AN","LA","CY","SE","CR","ED"}:
                        is_tag = True
                        tag_code = code_cand
                        content_remainder = match.group(2).strip() if match.group(2) else ""

                if is_tag:
                    # Save the previous tag's data before switching
                    if cur_tag and article_idx >= 0:
                        full_text = " ".join(cur_val).strip()
                        if full_text:
                            rows.append((article_idx, cur_tag, full_text))
                    
                    # If this is a Headline (HD), it signals a NEW article
                    if tag_code == "HD":
                        article_idx += 1
                        # Reset for new article
                        cur_tag = "HD"
                        cur_val = [content_remainder] if content_remainder else []
                    else:
                        # Just switching tags within the same article
                        cur_tag = tag_code
                        cur_val = [content_remainder] if content_remainder else []
                
                else:
                    # It's not a tag line, so it belongs to the current tag (continuation text)
                    if cur_tag and article_idx >= 0:
                        cur_val.append(line)
        
        # End of file: save the very last tag buffer
        if cur_tag and article_idx >= 0:
            full_text = " ".join(cur_val).strip()
            if full_text:
                rows.append((article_idx, cur_tag, full_text))

    return rows

def parse_folder(pdf_dir):
    """Run parse_pdf_linear on all PDFs in a folder."""
    data = []
    files = sorted(glob.glob(os.path.join(pdf_dir, "*.pdf")))
    if not files:
        print(f"WARNING: No PDF files found in {pdf_dir}")
        
    for path in files:
        print(f"Parsing: {os.path.basename(path)}...")
        rows = parse_pdf_linear(path)
        base = os.path.basename(path)
        for art_idx, code, text in rows:
            data.append((base, art_idx, code, text))
            
    df = pd.DataFrame(data, columns=["pdf_file","article_index","code","text"])
    return df

def to_wide(df_long):
    if df_long.empty:
        return pd.DataFrame()
    # Deduplicate just in case
    df_long = df_long.drop_duplicates(subset=["pdf_file","article_index","code"])
    
    wide = df_long.pivot_table(index=["pdf_file","article_index"], columns="code",
                               values="text", aggfunc="first")
    wide = wide.reset_index()
    wide.columns.name = None
    return wide

# ====================== LLM / PROCESSING ==========================

def make_prompt(brand: str, aliases, row) -> str:
    hd = str(row.get("HD","") or "")
    sn = str(row.get("SN","") or "")
    pd_ = str(row.get("PD","") or "")

    # Gather text from body (TD) and lead paragraph (LP) if available
    potential_cols = ["LP", "TD", "CO", "IN", "NS", "RE"]
    text_parts = []
    for col in potential_cols:
        val = str(row.get(col, "") or "").strip()
        if len(val) > 2: 
            text_parts.append(val)
    
    text_body = " \n ".join(text_parts)
    # Truncate to avoid 500 errors, but keep enough for context
    text_body = text_body[:6000] 

    alias_str = ", ".join(sorted(set([brand] + list(aliases))))
    
    prompt = f"""
You are a financial and news analyst. Classify the article below.

Target Brand: "{brand}"
Aliases: {alias_str}

Metadata:
- Headline: {hd}
- Source: {sn}
- Date: {pd_}

Article Snippet:
{text_body}

TASKS:
1. Is this article RELEVANT to "{brand}"? (True/False)
2. Does this article mention "Rana Plaza" or the 2013 factory collapse? (Yes/No)

CRITERIA (Brand Relevant = TRUE):
- Headline mentions brand.
- Corporate news (stocks, earnings, deals, CEO).
- Product launches or marketing/ads involving the brand.
- CSR/donations by the brand.

CRITERIA (Brand Relevant = FALSE):
- Shopping List: Brand is just one item in a list of many.
- Same Name: Refers to a person (e.g. "Shan Hanes").
- Passing Mention: Minor mention in unrelated story.

OUTPUT JSON ONLY:
{{
  "brand_relevant": true,
  "Mention_of_Rana_Plaza": "No"
}}
"""
    return textwrap.dedent(prompt).strip()

def call_ollama(prompt: str) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_ctx": 4096 
        },
    }
    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT_SEC)
        r.raise_for_status()
        return (r.json().get("response") or "").strip()
    except Exception as e:
        print(f"LLM Error: {e}")
        return ""

def safe_parse_json(raw: str) -> dict:
    raw = raw.strip()
    # Try to find JSON object
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        raw = match.group(0)

    try:
        return json.loads(raw)
    except:
        pass
    
    # Fallback Regex
    brand_rel = bool(re.search(r'[\'"]brand_relevant[\'"]\s*:\s*true', raw, re.IGNORECASE))
    
    rana_val = "No"
    if re.search(r'[\'"]Mention_of_Rana_Plaza[\'"]\s*:\s*[\'"]?Yes', raw, re.IGNORECASE):
        rana_val = "Yes"
        
    return {"brand_relevant": brand_rel, "Mention_of_Rana_Plaza": rana_val, "raw": raw}

def lang_audience_country(la: str) -> str | None:
    if not isinstance(la, str) or not la.strip(): return None
    la_norm = la.lower()
    for key, country in LANG_TO_COUNTRY.items():
        if key in la_norm: return country
    return None

# ========================= PIPELINE STEPS ===================================

def step1_parse_pdfs():
    print(f"[Step 1] Parsing PDFs for brand: {BRAND}")
    os.makedirs(BRAND_PDF_DIR, exist_ok=True)
    
    df_long = parse_folder(BRAND_PDF_DIR)
    if df_long.empty:
        print("ERROR: No data parsed. Check PDF location.")
        return

    df_wide = to_wide(df_long)
    
    df_long.to_csv(BRAND_LONG_CSV, index=False)
    df_wide.to_csv(BRAND_WIDE_CSV, index=False)
    print(f"Saved parsed data to: {BRAND_WIDE_CSV}")
    print(f"Total articles found: {len(df_wide)}")

def step2_llm_relevance():
    print(f"[Step 2] LLM classification for brand: {BRAND}")
    if not os.path.exists(BRAND_WIDE_CSV):
        print("Run Step 1 first!")
        return

    df = pd.read_csv(BRAND_WIDE_CSV, dtype=str)
    
    # Load cache if exists
    cache = {}
    if os.path.exists(BRAND_CACHE_JSON):
        with open(BRAND_CACHE_JSON,"r",encoding="utf-8") as f:
            cache = json.load(f)
            


    results = []
    
    print(f"Classifying {len(df)} articles...")
    
    for idx, row in df.iterrows():
        key = f"{row['pdf_file']}|{row['article_index']}"
        
        # Check cache
        cached_val = cache.get(key)
        if cached_val and "Mention_of_Rana_Plaza" in str(cached_val):
            parsed = cached_val if "raw" not in cached_val else safe_parse_json(cached_val["raw"])
        else:
            prompt = make_prompt(BRAND, BRAND_ALIASES, row)
            raw_resp = call_ollama(prompt)
            parsed = safe_parse_json(raw_resp)
            parsed["raw"] = raw_resp
            cache[key] = parsed
            print(f"Processed {idx+1}/{len(df)}...", end="\r")
            time.sleep(SLEEP_BETWEEN)

        results.append({
            "pdf_file": row["pdf_file"],
            "article_index": row["article_index"],
            "brand_relevant": parsed.get("brand_relevant", False),
            "Mention_of_Rana_Plaza": parsed.get("Mention_of_Rana_Plaza", "No"),
            "raw_llm": parsed.get("raw", "")
        })

    print("")
    # Save Cache
    with open(BRAND_CACHE_JSON,"w",encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

    out_df = pd.DataFrame(results)
    out_df.to_csv(BRAND_RELEV_CSV, index=False)
    
    print(f"Saved relevance labels -> {BRAND_RELEV_CSV}")
    print(f"Brand Relevant: {out_df['brand_relevant'].sum()}")
    print(f"Rana Plaza Mentions: {(out_df['Mention_of_Rana_Plaza']=='Yes').sum()}")

def step3_country_and_panel():
    print(f"\n[Step 3] Aggregation for brand: {BRAND}")
    
    if not os.path.exists(BRAND_WIDE_CSV) or not os.path.exists(BRAND_RELEV_CSV):
        print("Missing input files.")
        return

    dfw = pd.read_csv(BRAND_WIDE_CSV, dtype=str, keep_default_na=False)
    rel = pd.read_csv(BRAND_RELEV_CSV, dtype={"pdf_file":str,"article_index":str})
    
    # Merge
    dfw["article_index"] = dfw["article_index"].astype(str)
    merged = dfw.merge(rel, on=["pdf_file","article_index"], how="left")
    
    # Filter Relevant
    merged["brand_relevant"] = merged["brand_relevant"].astype(str).str.lower() == "true"
    final_df = merged[merged["brand_relevant"]].copy()
    
    # Map Countries
    if os.path.exists(MEDIA_COUNTRY_GLOBAL):
        map_df = pd.read_excel(MEDIA_COUNTRY_GLOBAL, dtype=str, keep_default_na=False)
        map_df.columns = map_df.columns.str.strip()
        # Find country col
        tgt_col = "country" if "country" in map_df.columns else "audience_country"
        sn_map = dict(zip(map_df["Source Code"].astype(str), map_df[tgt_col].astype(str)))
        
        # Apply map
        final_df["Country_global"] = final_df["SC"].map(sn_map).fillna(final_df["SN"].map(sn_map))
        
        # Apply Lang override
        final_df["Country_lang"] = final_df["LA"].apply(lang_audience_country)
        final_df["Country"] = final_df["Country_lang"].fillna(final_df["Country_global"])
        
        # Handle blanks
        final_df["Country"] = final_df["Country"].replace(r'^\s*$', "NO country mapping", regex=True).fillna("NO country mapping")
    else:
        print("WARNING: Country mapping Excel not found. Skipping country map.")
        final_df["Country"] = "Unknown"

    # Save detailed
    detailed_path = os.path.join(BRAND_PDF_DIR, f"{BRAND}_detailed_relevance_check.csv")
    final_df.to_csv(detailed_path, index=False)
    print(f"Saved detailed list -> {detailed_path}")

    # Aggregate
    final_df["dt"] = pd.to_datetime(final_df["PD"], dayfirst=True, errors="coerce")
    final_df["month"] = final_df["dt"].dt.month
    
    if YEAR_FILTER:
        final_df = final_df[final_df["dt"].dt.year == YEAR_FILTER]

    if final_df.empty:
        print("No relevant data for aggregation.")
        return

    grp = final_df.groupby(["Country","month"], as_index=False).agg(
        Number_of_Articles=("brand_relevant", "count"),
        Rana_Plaza_Mentions=("Mention_of_Rana_Plaza", lambda x: (x=="Yes").sum())
    )
    
    grp.insert(0, "Brand", BRAND)
    grp.rename(columns={"month":"Time"}, inplace=True)
    
    grp.to_csv(BRAND_COUNTRY_CSV, index=False)
    grp.to_excel(BRAND_FINAL_XLSX, index=False)
    print(f"Saved Panel -> {BRAND_COUNTRY_CSV}")
    print(grp.head())

if __name__ == "__main__":

    
    step1_parse_pdfs()      
    step2_llm_relevance()   
    step3_country_and_panel()

In [ ]:
import os
if os.path.exists(BRAND_CACHE_JSON):
    os.remove(BRAND_CACHE_JSON)
    print("Deleted cache.")